## Analysis process - Dianela

In [65]:
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

(17150, 57)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,EMPLOYEES,BUSINESSES,gva_per_hour,weekly_pay,employment_rate,unemployment_rate,...,lq_bus,emp_share,lq_emp,growth_emp,cagr_emp,growth_bus,cagr_bus,related_variety,size_large_share,size_micro_share
0,2016,E06000001,Hartlepool,Advanced Manufacturing,1470.0,25.0,29.84,521.2,69.7,4.6,...,0.840582,0.048197,1.661003,0.010204,0.001693,-0.200000,-0.036508,1.332179,0.000000,1.000000
1,2016,E06000001,Hartlepool,Creative Industries,450.0,110.0,29.84,521.2,69.7,4.6,...,0.355425,0.014754,0.297588,0.744444,0.097176,-0.090909,-0.015760,2.200516,0.000000,1.000000
2,2016,E06000001,Hartlepool,Defence,0.0,0.0,29.84,521.2,69.7,4.6,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
3,2016,E06000001,Hartlepool,Digital and Technologies,1510.0,440.0,29.84,521.2,69.7,4.6,...,1.362619,0.049508,0.701351,-0.076159,-0.013116,-0.352273,-0.069823,1.072313,0.000000,1.000000
4,2016,E06000001,Hartlepool,Financial Services,285.0,40.0,29.84,521.2,69.7,4.6,...,0.383770,0.009344,0.157782,-0.070175,-0.012053,0.375000,0.054509,1.213008,0.000000,0.750000
5,2016,E06000001,Hartlepool,Life Sciences,40.0,0.0,29.84,521.2,69.7,4.6,...,0.000000,0.001311,0.456144,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN
6,2016,E06000001,Hartlepool,Professional and Business Services,2630.0,870.0,29.84,521.2,69.7,4.6,...,0.991174,0.086230,0.483176,-0.019011,-0.003194,-0.264368,-0.049884,1.830641,0.000000,0.965517
7,2016,E06000002,Middlesbrough,Advanced Manufacturing,620.0,25.0,29.50,481.9,68.8,5.1,...,0.587201,0.010622,0.366062,0.153226,0.024045,0.400000,0.057681,1.332179,0.000000,0.800000
8,2016,E06000002,Middlesbrough,Creative Industries,1320.0,180.0,29.50,481.9,68.8,5.1,...,0.406288,0.022614,0.456128,0.431818,0.061650,0.055556,0.009052,2.328951,0.000000,1.000000
9,2016,E06000002,Middlesbrough,Defence,0.0,0.0,29.50,481.9,68.8,5.1,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN


In [ ]:
# ─────────────────────────────────────────────
# 2. DROP METADATA COLUMNS
# ─────────────────────────────────────────────
drop_patterns = [
    'Local Authority District', 'Observation Status', 'Data accuracy',
    'Country', 'Nation', 'Total value of UK', 'Total UK',
    '% living in', 'Homicide Offences', '_merge', 'Area Code',
    'County_or_Unitary_Authority',
]
df = df.drop(columns=[c for c in df.columns
                       if any(c.startswith(p) for p in drop_patterns)])

# ─────────────────────────────────────────────
# 3. DEFINE INDICATOR COLUMNS
#    Excludes reverse-causality and no-EEG-mechanism
#    variables per codebook
# ─────────────────────────────────────────────
ID_COLS      = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 'IS8_SECTOR']
OUTCOME_COLS = ['OBS_Value_Business', 'OBS_Value_Employment', 'growth_firms', 'growth_emp']

EXCLUDED = [
    # Reverse causality — outcomes of IS8 presence
    'Gross Value Added (GVA) per hour worked (£) [GVA per hour]',
    'Gross median weekly pay (£) [Weekly pay]',
    'Gross disposable household income, per head (£) [GDHI per head]',
    'Employment rate, aged 16 to 64 years (%) [Employment rate]',
    'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]',
    # Health — no EEG mechanism
    'Percentage of adults that currently smoke cigarettes (%) [Smokers]',
    'Proportion of children living with obesity at reception age (%) [Reception obesity]',
    'Proportion of children living with obesity at Year 6 age (%) [Year 6 obesity]',
    'Proportion of adults living with obesity, aged 18 years and over (%) [Adult obesity]',
    'Proportion of cancers diagnosed at stages 1 and 2 (%) [Cancer diagnosis]',
    'Age-standardised mortality rate for those aged under 75 (per 100,000 population) [Under 75 mortality rate]',
    # Wellbeing — outcomes, not enablers
    'Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction]',
    'Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile]',
    'Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness]',
    'Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]',
    # Early years — 15-20yr lag
    "Percentage of 5-year olds at 'expected level' in communication and language early learning goals (%) [Early years comms]",
    "Percentage of 5-year olds at 'expected level' in literacy early learning goals (%) [Early years literacy]",
    "Percentage of 5-year olds at 'expected level' in maths early learning goals (%) [Early years maths]",
    # Below LAD resolution
    'Percentage of pupils in state-funded schools meeting the expected standard in reading, writing and maths at the end of key stage 2 (%) [KS2 attainment]',
    'Percentage of young people achieving GCSEs (and equivalent qualifications) in English and maths by age 19 (%) [GCSE by age 19]',
    'Percentage of schools rated good or outstanding by Ofsted (%) [Ofsted]',
    'Percentage of persistent absences (10% or more missed) for all pupils (%) [Persistent absences]',
    'Percentage of persistent absences (10% or more missed) eligible for free school meals in the past 6 years (%) [Persistent absences FSM]',
    'Percentage of persistent absences (10% or more missed) for pupils who are looked after continuously for at least 12 months by local authorities (%) [Persistent absences CLA]',
    'Count of 19+ Further Education and Skills Learner Achievements (qualifications) [FE and skills achievements]',
    'Female Healthy Life Expectancy (years) [Female HLE]',
    'Male Healthy Life Expectancy (years) [Male HLE]',
]

INDICATOR_COLS = [c for c in df.columns
                  if c not in ID_COLS + OUTCOME_COLS + EXCLUDED]

# Convert to numeric
for col in INDICATOR_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"\nIndicators retained ({len(INDICATOR_COLS)}):")
for c in INDICATOR_COLS:
    print(f"  {c}")

# ─────────────────────────────────────────────
# 4. BUILD LAD-LEVEL CONTROLS
#    (indicators repeat across sectors — collapse
#     to one row per LAD by taking the mean)
# ─────────────────────────────────────────────
x_lad = (
    df.groupby('GEOGRAPHY_CODE')[INDICATOR_COLS]
    .mean(numeric_only=True)
    .reset_index()
)



Indicators retained (14):
  Count of births of new enterprises (control rounded to base 5) [New enterprises]
  Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises]
  Count of active enterprises (control rounded to base 5) [Active enterprises]
  Count of high growth enterprises (control rounded to base 5) [High growth enterprises]
  Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by public transport or walking (minutes) [Public transport to employer]
  Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by car (minutes) [Drive to employer]
  Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by cycle (minutes) [Cycle to employer]
  Percentage of premises with gigabit-capable broadband (%) [Broadband availability]
  Percentage of 4G coverage by at least one mobile network operator (%) [4G area coverage]
 

In [55]:
# ─────────────────────────────────────────────
# 5. IMPUTE MISSING → LAD median
# ─────────────────────────────────────────────
for col in INDICATOR_COLS:
    x_lad[col] = x_lad[col].fillna(x_lad[col].median())

print(f"\nMissing after imputation: {x_lad[INDICATOR_COLS].isnull().sum().sum()}")


Missing after imputation: 0


In [56]:
# ─────────────────────────────────────────────
# 6. PIVOT OUTCOMES TO WIDE
#    growth_firms + growth_emp
#    One row per LAD, one column per IS8 sector
# ─────────────────────────────────────────────
def pivot_outcome(data, outcome_var, prefix):
    wide = (
        data[['GEOGRAPHY_CODE', 'IS8_SECTOR', outcome_var]]
        .pivot_table(index='GEOGRAPHY_CODE', columns='IS8_SECTOR',
                     values=outcome_var, aggfunc='first')
        .reset_index()
    )
    wide.columns.name = None
    wide.columns = (
        ['GEOGRAPHY_CODE'] +
        [prefix + c.replace(' ', '_') for c in wide.columns[1:]]
    )
    return wide

y_firms = pivot_outcome(df, 'growth_firms', 'gfirms_')
y_emp   = pivot_outcome(df, 'growth_emp',   'gemp_')


In [60]:
# ─────────────────────────────────────────────
# 7. MERGE CONTROLS + OUTCOMES
#    Final dataset: 355 rows × (indicators + sectors)
# ─────────────────────────────────────────────
lad_df = x_lad.merge(y_firms, on='GEOGRAPHY_CODE', how='inner')
lad_df = lad_df.merge(y_emp,   on='GEOGRAPHY_CODE', how='inner')
print(f"\nFinal LAD dataset: {lad_df.shape}")

gfirms_cols = [c for c in lad_df.columns if c.startswith('gfirms_')]
gemp_cols   = [c for c in lad_df.columns if c.startswith('gemp_')]
print(f"Business growth columns : {gfirms_cols}")
print(f"Employment growth columns: {gemp_cols}")

# ─────────────────────────────────────────────
# 8. STANDARDISE X
# ─────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(lad_df[INDICATOR_COLS].astype(float))


Final LAD dataset: (355, 29)
Business growth columns : ['gfirms_Advanced_manufacturing', 'gfirms_Creative_Industries', 'gfirms_Defence_sector', 'gfirms_Digital_and_Technology', 'gfirms_Financial_Services', 'gfirms_Life_Sciences', 'gfirms_Professional_and_Business_Services']
Employment growth columns: ['gemp_Advanced_manufacturing', 'gemp_Creative_Industries', 'gemp_Defence_sector', 'gemp_Digital_and_Technology', 'gemp_Financial_Services', 'gemp_Life_Sciences', 'gemp_Professional_and_Business_Services']


In [61]:
# ─────────────────────────────────────────────
# 9. LASSO — one model per IS8 sector
# ─────────────────────────────────────────────
# Short labels for display
short_labels = {c: c.split('[')[0].strip()[:55] for c in INDICATOR_COLS}

def run_lasso(X_sc, y_series, sector_name):
    """Fit LassoCV on non-null observations. Return selected vars + R²."""
    mask = y_series.notna()
    n    = mask.sum()
    if n < 30:
        print(f"\n  {sector_name}: skipped (only {n} non-null LADs)")
        return pd.Series(dtype=float), np.nan

    model = LassoCV(cv=10, random_state=42, max_iter=10000).fit(
        X_sc[mask], y_series[mask]
    )
    coefs    = pd.Series(model.coef_, index=INDICATOR_COLS)
    selected = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    r2       = model.score(X_sc[mask], y_series[mask])

    print(f"\n{'─'*60}")
    print(f"SECTOR: {sector_name}")
    print(f"  N LADs      : {n}")
    print(f"  Best alpha  : {model.alpha_:.5f}")
    print(f"  R² (train)  : {r2:.3f}")
    print(f"  Selected {len(selected)} variable(s):")
    if len(selected) == 0:
        print("    (none — all coefficients shrunk to zero)")
    for var, val in selected.items():
        print(f"    {val:+.4f}  {short_labels[var]}")

    return selected, r2

print("\n" + "═"*60)
print("LASSO: BUSINESS COUNT GROWTH (growth_firms) per IS8 sector")
print("═"*60)
results_firms = {}
for col in gfirms_cols:
    sector = col.replace('gfirms_', '').replace('_', ' ')
    sel, r2 = run_lasso(X_scaled, lad_df[col], sector)
    results_firms[sector] = {'selected': sel, 'r2': r2}

print("\n" + "═"*60)
print("LASSO: EMPLOYMENT GROWTH (growth_emp) per IS8 sector")
print("═"*60)
results_emp = {}
for col in gemp_cols:
    sector = col.replace('gemp_', '').replace('_', ' ')
    sel, r2 = run_lasso(X_scaled, lad_df[col], sector)
    results_emp[sector] = {'selected': sel, 'r2': r2}


════════════════════════════════════════════════════════════
LASSO: BUSINESS COUNT GROWTH (growth_firms) per IS8 sector
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
SECTOR: Advanced manufacturing
  N LADs      : 355
  Best alpha  : 0.01885
  R² (train)  : 0.015
  Selected 1 variable(s):
    -0.0098  Percentage of premises with gigabit-capable broadband (

────────────────────────────────────────────────────────────
SECTOR: Creative Industries
  N LADs      : 355
  Best alpha  : 0.00023
  R² (train)  : 0.092
  Selected 11 variable(s):
    +0.0255  Average travel time in minutes to reach nearest large e
    -0.0200  Average travel time in minutes to reach nearest large e
    -0.0120  Apprenticeships started by adults aged 16+ based on hom
    -0.0103  Percentage of premises with gigabit-capable broadband (
    +0.0067  19+ further education and skills participation (per 100
    +0.0067  Apprenticeships achieve

In [ ]:
# ─────────────────────────────────────────────
# 10. SUMMARY TABLE
#     How many sectors selected each variable?
# ─────────────────────────────────────────────
def build_summary(results_dict, label):
    rows = []
    for sector, res in results_dict.items():
        if isinstance(res['selected'], pd.Series):
            for var, coef in res['selected'].items():
                rows.append({'Sector': sector, 'Variable': var, 'Coefficient': coef})
    if not rows:
        print(f"\nNo variables selected for {label}.")
        return pd.DataFrame()
    summary = pd.DataFrame(rows)
    freq = (
        summary
        .groupby('Variable')
        .agg(
            N_sectors  = ('Sector',      'count'),
            Mean_coef  = ('Coefficient', 'mean'),
            Sectors    = ('Sector',      lambda x: ', '.join(sorted(x)))
        )
        .sort_values('N_sectors', ascending=False)
        .reset_index()
    )
    freq['Variable_short'] = freq['Variable'].map(short_labels)
    print("\n" + "═"*60)
    print(f"SELECTION FREQUENCY — {label}")
    print("═"*60)
    print(freq[['Variable_short', 'N_sectors', 'Mean_coef', 'Sectors']].to_string(index=False))
    return freq

freq_firms = build_summary(results_firms, "Business Growth (growth_firms)")
freq_emp   = build_summary(results_emp,   "Employment Growth (growth_emp)")

# ─────────────────────────────────────────────
# 11. PLOT — coefficient heatmap
# ─────────────────────────────────────────────
def plot_heatmap(results_dict, freq_df, outcome_label, fname):
    if freq_df.empty:
        return
    all_vars = freq_df['Variable'].tolist()
    sectors  = list(results_dict.keys())
    mat = pd.DataFrame(0.0,
                       index=[short_labels[v] for v in all_vars],
                       columns=sectors)
    for sector, res in results_dict.items():
        if isinstance(res['selected'], pd.Series):
            for var, coef in res['selected'].items():
                if short_labels[var] in mat.index:
                    mat.loc[short_labels[var], sector] = coef

    vmax = np.abs(mat.values).max() or 1
    fig, ax = plt.subplots(figsize=(max(10, len(sectors) * 1.6),
                                    max(4,  len(all_vars) * 0.55 + 1.5)))
    im = ax.imshow(mat.values, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(sectors)))
    ax.set_xticklabels([s.replace(' ', '\n') for s in sectors], fontsize=9)
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=9)
    plt.colorbar(im, ax=ax, label='LASSO coefficient (standardised X)')
    ax.set_title(f'LASSO — {outcome_label} by IS8 Sector\n'
                 'Red = positive, Blue = negative, White = not selected',
                 fontsize=10, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

plot_heatmap(results_firms, freq_firms,
             'Business Count Growth (growth_firms)', 'lasso_heatmap_growth_firms.png')
plot_heatmap(results_emp,   freq_emp,
             'Employment Growth (growth_emp)',        'lasso_heatmap_growth_emp.png')

print("\n✓ Done.")
print(f"  LADs: {len(lad_df)}  |  Indicators: {len(INDICATOR_COLS)}  |  Models: {len(gfirms_cols) + len(gemp_cols)}")

In [19]:
cols_to_drop = [col for col in analysis_data.columns if 
                col.startswith('Local Authority District') or
                col.startswith('Notes') or
                col.startswith('Data accuracy') or
                col.startswith('Observation Status') or
                col.startswith('Country') or
                col.startswith('Nation') or
                col.startswith('Total value of UK') or               
                col.startswith('Total UK') or 
                col.startswith('% living in') or 
                col.startswith('Homicide Offences')]
# Drop merged variable and pattern-matched columns
analysis_data = analysis_data.drop(columns=['_merge', 'Area Code'] + cols_to_drop)

print(f"Columns dropped: {len(cols_to_drop) + 2}")
print(f"Remaining columns: {analysis_data.shape[1]}")

Columns dropped: 60
Remaining columns: 50


In [20]:
# Columns to exclude from indicators
cols_to_exclude = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 
                   'IS8_SECTOR','OBS_Value_Business', 'OBS_Value_Employment',
                   'growth_firms', 'growth_emp']

x = analysis_data.drop(columns=cols_to_exclude)
y = analysis_data[cols_to_exclude]

In [21]:
#With the previous code, I realised that some of the indicators were not numeric, so I will convert them to numeric:
cols_to_convert = [
    'Gross median weekly pay (£) [Weekly pay]',
    'Employment rate, aged 16 to 64 years (%) [Employment rate]',
    'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]',
    'Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]',
    'Percentage of adults that currently smoke cigarettes (%) [Smokers]',
    'Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction]',
    'Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile]',
    'Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness]',
    'Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]'
]
for col in cols_to_convert:
    x[col] = pd.to_numeric(x[col], errors='coerce')

print(x.dtypes)

County_or_Unitary_Authority                                                                                                                                                       object
Gross Value Added (GVA) per hour worked (£) [GVA per hour]                                                                                                                       float64
Gross median weekly pay (£) [Weekly pay]                                                                                                                                         float64
Employment rate, aged 16 to 64 years (%) [Employment rate]                                                                                                                       float64
Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]                                                                                                       float64
Gross disposable household income, per head (£) [GDHI per head]            

In [22]:
x_controls = x.groupby("County_or_Unitary_Authority").mean(numeric_only=True).reset_index()
x_controls.head(10)

,County_or_Unitary_Authority,Gross Value Added (GVA) per hour worked (£) [GVA per hour],Gross median weekly pay (£) [Weekly pay],"Employment rate, aged 16 to 64 years (%) [Employment rate]","Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]","Gross disposable household income, per head (£) [GDHI per head]",Count of births of new enterprises (control rounded to base 5) [New enterprises],Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises],Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],...,Percentage of adults that currently smoke cigarettes (%) [Smokers],Proportion of children living with obesity at reception age (%) [Reception obesity],Proportion of children living with obesity at Year 6 age (%) [Year 6 obesity],"Proportion of adults living with obesity, aged 18 years and over (%) [Adult obesity]","Age-standardised mortality rate for those aged under 75 (per 100,000 population) [Under 75 mortality rate]",Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety],"Net additions per 1,000 stock [Net additions]"
0,Bath and North East Somerset,28.45,564.1,80.2,2.4,23101.0,835.0,945.0,8980.0,50.0,...,11.5,7.44048,17.30205,21.68340,20.12109,7.38,7.66,7.20,3.59,6.0
1,Bedford,32.74,625.3,77.6,4.5,21328.0,965.0,1025.0,8075.0,30.0,...,11.8,8.33333,21.47806,24.02529,29.30756,7.46,7.71,7.59,3.35,18.0
2,Blackburn with Darwen,28.48,496.0,67.9,4.7,15025.0,815.0,855.0,6270.0,35.0,...,19.4,10.35354,24.72406,22.97324,48.34618,7.24,7.80,7.43,3.07,9.0
3,Blackpool,28.32,463.7,71.1,4.2,16717.0,795.0,565.0,4670.0,20.0,...,18.8,11.98630,26.92308,35.62741,52.62189,7.49,7.64,7.31,3.50,3.0
4,"Bournemouth, Christchurch and Poole",34.79,545.9,77.9,NaN,21751.0,1935.0,1935.0,17220.0,75.0,...,10.1,6.90162,18.96104,27.92590,27.00597,7.57,7.73,7.37,3.10,4.0
5,Bracknell Forest,49.02,669.8,80.0,3.0,23748.0,520.0,565.0,5040.0,30.0,...,14.6,7.24638,19.41392,24.98433,26.02915,7.55,7.74,7.67,3.41,8.0
6,Brighton and Hove,39.36,572.3,72.4,4.7,24834.0,2165.0,2200.0,16810.0,55.0,...,12.8,6.80751,17.02586,19.12000,27.78213,7.39,7.67,7.03,3.31,7.0
7,"Bristol, City of",33.63,591.2,76.7,3.5,21084.0,2540.0,2380.0,20545.0,115.0,...,14.8,8.91089,21.59827,24.82034,29.88635,7.31,7.33,7.20,3.44,8.0
8,Buckinghamshire,38.65,640.5,80.5,NaN,28440.0,3075.0,3620.0,33810.0,100.0,...,10.6,7.11150,17.12219,20.51092,21.06846,7.63,7.84,7.47,3.26,16.0
9,Central Bedfordshire,34.00,659.8,82.8,NaN,22509.0,1315.0,1490.0,13060.0,35.0,...,15.0,6.24093,17.98246,30.36404,23.27008,7.32,7.75,7.28,3.21,18.0


In [23]:
y.head(25)

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,OBS_Value_Business,OBS_Value_Employment,growth_firms,growth_emp
0,2022,E06000001,Hartlepool,Advanced manufacturing,20.0,1485.0,-0.213574,0.295338
1,2022,E06000001,Hartlepool,Creative Industries,100.0,785.0,-0.094410,0.032323
2,2022,E06000001,Hartlepool,Defence sector,0.0,0.0,0.000000,0.000000
3,2022,E06000001,Hartlepool,Digital and Technology,285.0,1395.0,-0.146126,-0.055725
4,2022,E06000001,Hartlepool,Financial Services,55.0,265.0,0.000000,0.255620
5,2022,E06000001,Hartlepool,Life Sciences,0.0,40.0,0.000000,0.000000
6,2022,E06000001,Hartlepool,Professional and Business Services,640.0,2580.0,-0.103643,-0.264062
7,2022,E06000002,Middlesbrough,Advanced manufacturing,35.0,715.0,0.325423,0.555087
8,2022,E06000002,Middlesbrough,Creative Industries,190.0,1890.0,-0.051031,0.160251
9,2022,E06000002,Middlesbrough,Defence sector,0.0,0.0,0.000000,0.000000


In [24]:
y_pivot = y.pivot_table(
    index=['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME'],
    columns='IS8_SECTOR',
    values=['OBS_Value_Business', 'OBS_Value_Employment', 'growth_firms', 'growth_emp'],
    aggfunc='first'
).reset_index()

# Flatten MultiIndex columns
y_pivot.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) else col
    for col in y_pivot.columns
]
y_pivot.columns.name = None

# Rename prefixes to shorter versions
y_pivot.columns = (y_pivot.columns
    .str.replace('OBS_Value_Business_', 'count_bus_')
    .str.replace('OBS_Value_Employment_', 'count_em_')
    .str.replace('growth_firms_', 'gfirms_')
    .str.replace('growth_emp_', 'gemp_')
    .str.replace(' ', '_')
)
print(y_pivot.shape)
y_pivot.head()

(355, 31)


,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,count_bus_Advanced_manufacturing,count_bus_Creative_Industries,count_bus_Defence_sector,count_bus_Digital_and_Technology,count_bus_Financial_Services,count_bus_Life_Sciences,count_bus_Professional_and_Business_Services,...,gemp_Financial_Services,gemp_Life_Sciences,gemp_Professional_and_Business_Services,gfirms_Advanced_manufacturing,gfirms_Creative_Industries,gfirms_Defence_sector,gfirms_Digital_and_Technology,gfirms_Financial_Services,gfirms_Life_Sciences,gfirms_Professional_and_Business_Services
0,2022,E06000001,Hartlepool,20.0,100.0,0.0,285.0,55.0,0.0,640.0,...,0.255620,0.000000,-0.264062,-0.213574,-0.094410,0.0,-0.146126,0.000000,0.0,-0.103643
1,2022,E06000002,Middlesbrough,35.0,190.0,0.0,425.0,105.0,0.0,1160.0,...,-0.210614,0.000000,-0.045670,0.325423,-0.051031,0.0,-0.131769,0.000000,0.0,-0.070657
2,2022,E06000003,Redcar and Cleveland,35.0,145.0,0.0,360.0,65.0,0.0,875.0,...,0.150661,0.000000,-0.347651,-0.130053,0.187816,0.0,-0.053922,0.000000,0.0,-0.060893
3,2022,E06000004,Stockton-on-Tees,85.0,340.0,0.0,825.0,210.0,5.0,1990.0,...,-0.640111,-0.055263,-0.261752,0.123614,-0.043048,0.0,-0.092444,0.000000,0.0,-0.049006
4,2022,E06000005,Darlington,30.0,210.0,0.0,310.0,135.0,0.0,1005.0,...,0.206492,0.000000,-0.006210,0.000000,-0.090559,0.0,-0.120994,-0.036105,0.0,0.050979


In [28]:
fd_area_code = y_pivot.merge(
    x_controls,
    left_on='GEOGRAPHY_NAME',
    right_on='County_or_Unitary_Authority',
    how='left'
).drop(columns='County_or_Unitary_Authority')

print(fd_area_code.shape)
fd_area_code.head(50)

(355, 65)


,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,count_bus_Advanced_manufacturing,count_bus_Creative_Industries,count_bus_Defence_sector,count_bus_Digital_and_Technology,count_bus_Financial_Services,count_bus_Life_Sciences,count_bus_Professional_and_Business_Services,...,Percentage of adults that currently smoke cigarettes (%) [Smokers],Proportion of children living with obesity at reception age (%) [Reception obesity],Proportion of children living with obesity at Year 6 age (%) [Year 6 obesity],"Proportion of adults living with obesity, aged 18 years and over (%) [Adult obesity]","Age-standardised mortality rate for those aged under 75 (per 100,000 population) [Under 75 mortality rate]",Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety],"Net additions per 1,000 stock [Net additions]"
0,2022,E06000001,Hartlepool,20.0,100.0,0.0,285.0,55.0,0.0,640.0,...,14.3,12.68293,27.46781,35.35587,37.85297,7.29,7.75,7.57,3.48,11.0
1,2022,E06000002,Middlesbrough,35.0,190.0,0.0,425.0,105.0,0.0,1160.0,...,16.5,12.84153,28.53470,35.29352,40.03257,7.35,7.64,7.05,3.07,9.0
2,2022,E06000003,Redcar and Cleveland,35.0,145.0,0.0,360.0,65.0,0.0,875.0,...,13.7,9.44882,24.67949,34.80189,34.35631,7.25,7.67,7.49,3.25,7.0
3,2022,E06000004,Stockton-on-Tees,85.0,340.0,0.0,825.0,210.0,5.0,1990.0,...,13.2,9.71564,26.26263,32.84926,28.35783,7.33,7.62,7.29,3.23,7.0
4,2022,E06000005,Darlington,30.0,210.0,0.0,310.0,135.0,0.0,1005.0,...,11.5,11.93416,24.20635,29.99722,30.46435,7.32,7.63,7.17,3.38,10.0
5,2022,E06000006,Halton,85.0,185.0,0.0,420.0,110.0,15.0,1030.0,...,13.3,11.61049,28.00000,35.98169,42.19316,7.49,7.79,7.45,3.35,6.0
6,2022,E06000007,Warrington,90.0,740.0,0.0,1510.0,275.0,5.0,3895.0,...,9.9,9.06921,22.15686,29.54306,35.18737,7.45,7.69,7.32,3.18,16.0
7,2022,E06000008,Blackburn with Darwen,90.0,245.0,0.0,330.0,240.0,10.0,1355.0,...,19.4,10.35354,24.72406,22.97324,48.34618,7.24,7.80,7.43,3.07,9.0
8,2022,E06000009,Blackpool,25.0,245.0,0.0,270.0,135.0,5.0,685.0,...,18.8,11.98630,26.92308,35.62741,52.62189,7.49,7.64,7.31,3.50,3.0
9,2022,E06000010,"Kingston upon Hull, City of",170.0,340.0,0.0,430.0,230.0,15.0,1430.0,...,18.9,11.54472,26.75439,39.37604,48.33016,7.38,7.73,7.32,3.51,5.0


In [ ]:
threshold = 0.01  # drop columns with more than 1% missing
missing_pct = fd_area_code.isnull().mean()  # gives proportion (0 to 1)
cols_to_keep = missing_pct[missing_pct < threshold].index.tolist()
cols_to_drop = missing_pct[missing_pct >= threshold].index.tolist()

print(f"Columns kept:    {len(cols_to_keep)}")
print(f"Columns dropped: {len(cols_to_drop)}")
print(f"\nDropped columns:\n{cols_to_drop}")

# Apply
X_clean = fd_area_code[cols_to_keep]

X_clean.head()

Columns kept:    31
Columns dropped: 34

Dropped columns:
['Gross Value Added (GVA) per hour worked (£) [GVA per hour]', 'Gross median weekly pay (£) [Weekly pay]', 'Employment rate, aged 16 to 64 years (%) [Employment rate]', 'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]', 'Gross disposable household income, per head (£) [GDHI per head]', 'Count of births of new enterprises (control rounded to base 5) [New enterprises]', 'Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises]', 'Count of active enterprises (control rounded to base 5) [Active enterprises]', 'Count of high growth enterprises (control rounded to base 5) [High growth enterprises]', 'Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by public transport or walking (minutes) [Public transport to employer]', 'Average travel time in minutes to reach nearest large employment centre (500 to 4999 jobs available), by car (

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,count_bus_Advanced_manufacturing,count_bus_Creative_Industries,count_bus_Defence_sector,count_bus_Digital_and_Technology,count_bus_Financial_Services,count_bus_Life_Sciences,count_bus_Professional_and_Business_Services,...,gemp_Financial_Services,gemp_Life_Sciences,gemp_Professional_and_Business_Services,gfirms_Advanced_manufacturing,gfirms_Creative_Industries,gfirms_Defence_sector,gfirms_Digital_and_Technology,gfirms_Financial_Services,gfirms_Life_Sciences,gfirms_Professional_and_Business_Services
0,2022,E06000001,Hartlepool,20.0,100.0,0.0,285.0,55.0,0.0,640.0,...,0.255620,0.000000,-0.264062,-0.213574,-0.094410,0.0,-0.146126,0.000000,0.0,-0.103643
1,2022,E06000002,Middlesbrough,35.0,190.0,0.0,425.0,105.0,0.0,1160.0,...,-0.210614,0.000000,-0.045670,0.325423,-0.051031,0.0,-0.131769,0.000000,0.0,-0.070657
2,2022,E06000003,Redcar and Cleveland,35.0,145.0,0.0,360.0,65.0,0.0,875.0,...,0.150661,0.000000,-0.347651,-0.130053,0.187816,0.0,-0.053922,0.000000,0.0,-0.060893
3,2022,E06000004,Stockton-on-Tees,85.0,340.0,0.0,825.0,210.0,5.0,1990.0,...,-0.640111,-0.055263,-0.261752,0.123614,-0.043048,0.0,-0.092444,0.000000,0.0,-0.049006
4,2022,E06000005,Darlington,30.0,210.0,0.0,310.0,135.0,0.0,1005.0,...,0.206492,0.000000,-0.006210,0.000000,-0.090559,0.0,-0.120994,-0.036105,0.0,0.050979


In [ ]:
#Variables with growth rates:
X_clean= X_clean.rename(columns={
    'growth_firms': 'Business_growth_rate',
    'growth_emp': 'Employment_growth_rate'
})

In [ ]:
X_clean.head()

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,Business_growth_rate,Employment_growth_rate,Gross Value Added (GVA) per hour worked (£) [GVA per hour],...,Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],Percentage of premises with gigabit-capable broadband (%) [Broadband availability],Percentage of 4G coverage by at least one mobile network operator (%) [4G area coverage],Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications],Percentage of adults that currently smoke cigarettes (%) [Smokers],Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]
0,2022,E06000001,Hartlepool,Professional and Business Services,"Accounting, audit and tax consultancy",25.0,175.0,0.000000,0.839751,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
1,2022,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,15.0,0.000000,-1.558145,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
2,2022,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
3,2022,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
4,2022,E06000001,Hartlepool,Financial Services,Asset management and wholesale services,10.0,45.0,-0.374693,0.245122,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48


In [ ]:
#With the previous code, I realised that some of the indicators were not numeric, so I will convert them to numeric:
cols_to_convert = [
    'Gross median weekly pay (£) [Weekly pay]',
    'Employment rate, aged 16 to 64 years (%) [Employment rate]',
    'Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]',
    'Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]',
    'Percentage of adults that currently smoke cigarettes (%) [Smokers]',
    'Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction]',
    'Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile]',
    'Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness]',
    'Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]'
]
for col in cols_to_convert:
    X_clean[col] = pd.to_numeric(X_clean[col], errors='coerce')

print(X_clean.dtypes)

YEAR                                                                                                       int64
GEOGRAPHY_CODE                                                                                            object
GEOGRAPHY_NAME                                                                                            object
IS8_SECTOR                                                                                                object
FRONTIER_SECTOR                                                                                           object
OBS_Value_Business                                                                                       float64
OBS_Value_Employment                                                                                     float64
Business_growth_rate                                                                                     float64
Employment_growth_rate                                                                          

In [ ]:
##Number of missing values after conversion
print(X_clean.isnull().sum())

YEAR                                                                                                       0
GEOGRAPHY_CODE                                                                                             0
GEOGRAPHY_NAME                                                                                             0
IS8_SECTOR                                                                                                 0
FRONTIER_SECTOR                                                                                            0
OBS_Value_Business                                                                                         0
OBS_Value_Employment                                                                                       0
Business_growth_rate                                                                                       0
Employment_growth_rate                                                                                     0
Gross Value Added (

In [ ]:
total = len(X_clean)
missing_pct = (X_clean.isnull().sum() / total * 100).round(1)
print(missing_pct[missing_pct > 0])

Gross Value Added (GVA) per hour worked (£) [GVA per hour]                                               2.5
Gross median weekly pay (£) [Weekly pay]                                                                 0.6
Employment rate, aged 16 to 64 years (%) [Employment rate]                                               1.7
Modelled unemployment rate, aged 16 years and over (%) [Unemployment rate]                               5.9
Gross disposable household income, per head (£) [GDHI per head]                                          1.1
Count of births of new enterprises (control rounded to base 5) [New enterprises]                         1.1
Count of deaths of enterprises (control rounded to base 5) [Deaths of enterprises]                       1.1
Count of active enterprises (control rounded to base 5) [Active enterprises]                             1.1
Count of high growth enterprises (control rounded to base 5) [High growth enterprises]                   1.1
Percentage of premi

In [ ]:
# How many LADs needed the national median fallback?
lads_with_all_missing = X_clean.groupby('GEOGRAPHY_CODE')[col].apply(
    lambda x: x.isnull().all()
).sum()

print(f"LADs with no data at all for {col}: {lads_with_all_missing}")

LADs with no data at all for Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]: 16


One important thing — the wellbeing variables (Life satisfaction, Happiness, Worthwhile, Anxiety) and Unemployment rate tend to be missing in the same 16-21 LADs. This is likely because smaller or more rural LADs don't report these statistics.

For approximately 16-21 LADs (~5%) where no local data was available, missing values were imputed using the national median. These LADs are predominantly smaller or rural authorities with limited data reporting.

In [ ]:
# Columns to exclude from indicators
cols_to_exclude = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 
                   'IS8_SECTOR', 'FRONTIER_SECTOR', 
                   'OBS_Value_Business', 'OBS_Value_Employment',
                   'Business_growth_rate', 'Employment_growth_rate']

# X — only local indicators
indicators = X_clean.drop(columns=cols_to_exclude)

In [ ]:
##Imputations
fallback_summary = {}
for col in indicators.columns:
    lads_all_missing = X_clean.groupby('GEOGRAPHY_CODE')[col].apply(
        lambda x: x.isnull().all()
    ).sum()
    fallback_summary[col] = lads_all_missing
    
fallback_df = pd.DataFrame.from_dict(
    fallback_summary, orient='index', columns=['LADs using national median']
)

fallback_df = fallback_df[fallback_df['LADs using national median'] > 0].sort_values(
    'LADs using national median', ascending=False
)
print(fallback_df)
print(f"Total LADs in dataset: {X_clean['GEOGRAPHY_CODE'].nunique()}")

                                                    LADs using national median
Modelled unemployment rate, aged 16 years and o...                          21
Mean satisfaction with your life nowadays score...                          16
Mean feeling things done in life are worthwhile...                          16
Mean happiness yesterday scored 0 (not at all) ...                          16
Mean anxiety yesterday scored 0 (not at all) - ...                          16
Gross Value Added (GVA) per hour worked (£) [GV...                           9
Percentage of 4G coverage by at least one mobil...                           9
Percentage of premises with gigabit-capable bro...                           9
Percentage of adults that currently smoke cigar...                           7
Employment rate, aged 16 to 64 years (%) [Emplo...                           6
Proportion of the population aged 16 to 64 with...                           6
Count of births of new enterprises (control rou...  

In [ ]:
## I will drop of my analysis modelled unemployment rate, as I have the employment variable:




In [ ]:
for col in indicators.columns:
    X_clean[col] = X_clean.groupby('GEOGRAPHY_CODE')[col].transform(
        lambda x: x.fillna(x.median())
    )
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())  # fallback for any remaining missing with the national median 

In [ ]:
X_clean.head(50)

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,FRONTIER_SECTOR,OBS_Value_Business,OBS_Value_Employment,Business_growth_rate,Employment_growth_rate,Gross Value Added (GVA) per hour worked (£) [GVA per hour],...,Count of active enterprises (control rounded to base 5) [Active enterprises],Count of high growth enterprises (control rounded to base 5) [High growth enterprises],Percentage of premises with gigabit-capable broadband (%) [Broadband availability],Percentage of 4G coverage by at least one mobile network operator (%) [4G area coverage],Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications],Percentage of adults that currently smoke cigarettes (%) [Smokers],Mean satisfaction with your life nowadays scored 0 (not at all) - 10 (completely) [Life satisfaction],Mean feeling things done in life are worthwhile scored 0 (not at all) - 10 (completely) [Worthwhile],Mean happiness yesterday scored 0 (not at all) - 10 (completely) [Happiness],Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]
0,2022,E06000001,Hartlepool,Professional and Business Services,"Accounting, audit and tax consultancy",25.0,175.0,0.000000,0.839751,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
1,2022,E06000001,Hartlepool,Creative Industries,Advertising and marketing,10.0,15.0,0.000000,-1.558145,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
2,2022,E06000001,Hartlepool,Advanced manufacturing,Aerospace manufacturing,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
3,2022,E06000001,Hartlepool,Advanced manufacturing,Agritech,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
4,2022,E06000001,Hartlepool,Financial Services,Asset management and wholesale services,10.0,45.0,-0.374693,0.245122,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
5,2022,E06000001,Hartlepool,Advanced manufacturing,Automotive manufacturing,0.0,400.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
6,2022,E06000001,Hartlepool,Advanced manufacturing,Batteries,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
7,2022,E06000001,Hartlepool,Financial Services,Capital markets and retail investment,20.0,90.0,0.271934,0.056512,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
8,2022,E06000001,Hartlepool,Defence sector,Defence Sector,0.0,0.0,0.000000,0.000000,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48
9,2022,E06000001,Hartlepool,Digital and Technology,Digital and Technology,285.0,1395.0,-0.146126,-0.055725,29.84,...,2430.0,10.0,91.4,99.98,56.4,14.3,7.29,7.75,7.57,3.48


In [ ]:
lad_vars = (
    X_clean[X_clean["FRONTIER_SECTOR"] == "Total"]
    .groupby("GEOGRAPHY_CODE")
    .agg(
        GEOGRAPHY_NAME  = ("GEOGRAPHY_NAME", "first"),
        Business_growth = ("Business_growth_rate", "first"),  # adjust col name
        Employ_growth   = ("Employ_growth_rate",   "first"),  # adjust col name
        total_bus       = ("OBS_Value_Business", "first"),
        total_emp       = ("OBS_Value_Employment",  "first"),
    )
    .reset_index()
)

# 1b. Calculate sector shares from non-Total rows
sectors = X_clean[X_clean["FRONTIER_SECTOR"] != "Total"].copy()
sectors = sectors.merge(
    lad_vars[["GEOGRAPHY_CODE", "total_bus", "total_emp"]],
    on="GEOGRAPHY_CODE"
)

# Enterprise share per sector
sectors["ent_share"] = sectors["OBS_Value_Business"] / sectors["total_bus"]
# Employment share per sector
sectors["emp_share"] = sectors["OBS_Value_Employment"]  / sectors["total_emp"]

# 1c. Pivot to wide — one column per Frontier sector
ent_wide = sectors.pivot_table(
    index="GEOGRAPHY_CODE", columns="FRONTIER_SECTOR", values="ent_share"
)
ent_wide.columns = [f"ent_share_{c}" for c in ent_wide.columns]

emp_wide = sectors.pivot_table(
    index="GEOGRAPHY_CODE", columns="FRONTIER_SECTOR", values="emp_share"
)
emp_wide.columns = [f"emp_share_{c}" for c in emp_wide.columns]

# 1d. Merge everything together
df_wide = (
    lad_vars
    .set_index("GEOGRAPHY_CODE")
    .join(ent_wide)
    .join(emp_wide)
    .reset_index()
)

print(f"Wide dataset shape: {df_wide.shape}")  # should be ~350 rows
print(df_wide.head())


In [ ]:
ddddd

NameError: name 'ddddd' is not defined

In [ ]:
X = X_clean[indicators.columns].copy()
y_bus = X_clean['Business_growth_rate'].reset_index(drop=True)
y_emp = X_clean['Employment_growth_rate'].reset_index(drop=True)

print(f'X shape: {X.shape}, y_business growth shape: {y_bus.shape}, y_employment growth shape: {y_emp.shape}')

X shape: (7810, 17), y_business growth shape: (7810,), y_employment growth shape: (7810,)


In [ ]:
# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=indicators.columns, 
                            index=X_clean.index)

# Sector dummies
is8_dummies     = pd.get_dummies(X_clean['IS8_SECTOR'], 
                                  prefix='IS8', drop_first=True)
frontier_dummies = pd.get_dummies(X_clean['FRONTIER_SECTOR'], 
                                   prefix='FS', drop_first=True)

In [ ]:
cddddd

In [ ]:
##### LASSO — variable selection 
lasso_bus = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_scaled, y_bus)
lasso_emp = LassoCV(cv=10, random_state=42, max_iter=10000).fit(X_scaled, y_emp)

selected_bus = [col for col, coef in zip(indicators.columns, lasso_bus.coef_) if coef != 0]
selected_emp = [col for col, coef in zip(indicators.columns, lasso_emp.coef_) if coef != 0]

# LASSO coefficients summary
lasso_bus_df = pd.DataFrame({
    'Variable'   : indicators.columns,
    'Coefficient': lasso_bus.coef_
}).query('Coefficient != 0').sort_values('Coefficient', key=abs, ascending=False)

lasso_emp_df = pd.DataFrame({
    'Variable'   : indicators.columns,
    'Coefficient': lasso_emp.coef_
}).query('Coefficient != 0').sort_values('Coefficient', key=abs, ascending=False)


C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.903e+00, tolerance: 2.159e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.078e+01, tolerance: 2.159e-01
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\diane\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

In [ ]:
print(f"LASSO — Business growth selected {len(selected_bus)} variables:")
print(lasso_bus_df.to_string(index=False))

LASSO — Business growth selected 8 variables:
                                                                                             Variable  Coefficient
                                           Gross Value Added (GVA) per hour worked (£) [GVA per hour]    -0.009159
                                                             Gross median weekly pay (£) [Weekly pay]    -0.007358
    Proportion of the population aged 16 to 64 with NVQ3+ qualification (%) [Level 3+ qualifications]     0.006564
                             Mean anxiety yesterday scored 0 (not at all) - 10 (completely) [Anxiety]    -0.004866
                   Percentage of premises with gigabit-capable broadband (%) [Broadband availability]    -0.004678
                                   Percentage of adults that currently smoke cigarettes (%) [Smokers]     0.004444
                                           Employment rate, aged 16 to 64 years (%) [Employment rate]     0.002566
Mean satisfaction with your life n

In [ ]:
print(f"LASSO — Employment growth selected {len(selected_emp)} variables:")
print(lasso_emp_df.to_string(index=False))

LASSO — Employment growth selected 0 variables:
Empty DataFrame
Columns: [Variable, Coefficient]
Index: []


In [ ]:
# ── Sector dummies ────────────────────────────────────────────────
is8_dummies      = pd.get_dummies(X_clean['IS8_SECTOR'],      drop_first=True).astype(float)
frontier_dummies = pd.get_dummies(X_clean['FRONTIER_SECTOR'], drop_first=True).astype(float)

In [ ]:
# ── Modelos CON estandarización ───────────────────────────────────
def run_ols_scaled(y, dummies, label):
    X_ols = pd.concat([
        X_scaled_df[indicators.columns].reset_index(drop=True),
        dummies.reset_index(drop=True)
    ], axis=1).astype(float)
    X_ols = sm.add_constant(X_ols)
    model = sm.OLS(y, X_ols).fit(cov_type='HC3')
    print(f"\n{'='*60}")
    print(f"MODEL (SCALED): {label}")
    print(model.summary())
    return model

# ── Modelos SIN estandarización ───────────────────────────────────
def run_ols_raw(y, dummies, label):
    X_ols = pd.concat([
        X_clean[indicators.columns].reset_index(drop=True),
        dummies.reset_index(drop=True)
    ], axis=1).astype(float)
    X_ols = sm.add_constant(X_ols)
    model = sm.OLS(y, X_ols).fit(cov_type='HC3')
    print(f"\n{'='*60}")
    print(f"MODEL (RAW): {label}")
    print(model.summary())
    return model

### IS8 FIXED EFFECTS

Business outcome

In [ ]:

print("IS8 with standarized local indicators")
m1_scaled = run_ols_scaled(y_bus, is8_dummies, "M1: Business Growth   + IS8 FE")

IS8 with standarized local indicators

MODEL (SCALED): M1: Business Growth   + IS8 FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.253
Model:                              OLS   Adj. R-squared:                  0.250
Method:                   Least Squares   F-statistic:                     6.821
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):           1.38e-21
Time:                          15:45:56   Log-Likelihood:                -61410.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6011   BIC:                         1.230e+05
Df Model:                            23                                         
Covariance Type:                    HC3                                         
                                                                                                        

In [ ]:
print("IS8 without standarized local indicators")
m1_raw    = run_ols_raw   (y_bus, is8_dummies, "M1: Business Growth   + IS8 FE")

IS8 without standarized local indicators



MODEL (RAW): M1: Business Growth   + IS8 FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.253
Model:                              OLS   Adj. R-squared:                  0.250
Method:                   Least Squares   F-statistic:                     6.821
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):           1.38e-21
Time:                          15:45:56   Log-Likelihood:                -61410.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6011   BIC:                         1.230e+05
Df Model:                            23                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z      P>|z|

Employment outcome

In [ ]:
#With standarizaed local indicators
m2_scaled = run_ols_scaled(y_emp, is8_dummies, "M2: Employment Growth + IS8 FE")



MODEL (SCALED): M2: Employment Growth + IS8 FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.292
Method:                     Least Squares   F-statistic:                     11.13
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):           2.88e-40
Time:                            15:45:56   Log-Likelihood:                -73160.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6011   BIC:                         1.465e+05
Df Model:                              23                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std er

In [ ]:
#Without standarised local indicators
m2_raw    = run_ols_raw   (y_emp, is8_dummies, "M2: Employment Growth + IS8 FE")


MODEL (RAW): M2: Employment Growth + IS8 FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.292
Method:                     Least Squares   F-statistic:                     11.13
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):           2.88e-40
Time:                            15:45:56   Log-Likelihood:                -73160.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6011   BIC:                         1.465e+05
Df Model:                              23                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std err  

## FRONTIER SECTOR FIXED EFFECTS

Business outcome

In [ ]:
# With standarised local indicators
m3_scaled = run_ols_scaled(y_bus, frontier_dummies, "M3: Business Growth   + Frontier FE")



MODEL (SCALED): M3: Business Growth   + Frontier FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.255
Model:                              OLS   Adj. R-squared:                  0.251
Method:                   Least Squares   F-statistic:                     31.21
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):          7.68e-179
Time:                          15:45:56   Log-Likelihood:                -61405.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6001   BIC:                         1.231e+05
Df Model:                            33                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z   

In [ ]:
# Without standarised local indicators
m3_raw    = run_ols_raw   (y_bus, frontier_dummies, "M3: Business Growth   + Frontier FE")


MODEL (RAW): M3: Business Growth   + Frontier FE
                             OLS Regression Results                             
Dep. Variable:     Business_growth_rate   R-squared:                       0.255
Model:                              OLS   Adj. R-squared:                  0.251
Method:                   Least Squares   F-statistic:                     31.21
Date:                  Sun, 15 Mar 2026   Prob (F-statistic):          7.70e-179
Time:                          15:45:56   Log-Likelihood:                -61405.
No. Observations:                  6035   AIC:                         1.229e+05
Df Residuals:                      6001   BIC:                         1.231e+05
Df Model:                            33                                         
Covariance Type:                    HC3                                         
                                                                                                            coef    std err          z      

Employment outcome

In [ ]:
m4_scaled = run_ols_scaled(y_emp, frontier_dummies, "M4: Employment Growth + Frontier FE")


MODEL (SCALED): M4: Employment Growth + Frontier FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.291
Method:                     Least Squares   F-statistic:                     21.08
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):          1.10e-117
Time:                            15:45:57   Log-Likelihood:                -73159.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6001   BIC:                         1.466e+05
Df Model:                              33                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    s

In [ ]:
m4_raw    = run_ols_raw   (y_emp, frontier_dummies, "M4: Employment Growth + Frontier FE")


MODEL (RAW): M4: Employment Growth + Frontier FE
                              OLS Regression Results                              
Dep. Variable:     Employment_growth_rate   R-squared:                       0.295
Model:                                OLS   Adj. R-squared:                  0.291
Method:                     Least Squares   F-statistic:                     21.08
Date:                    Sun, 15 Mar 2026   Prob (F-statistic):          1.11e-117
Time:                            15:45:57   Log-Likelihood:                -73159.
No. Observations:                    6035   AIC:                         1.464e+05
Df Residuals:                        6001   BIC:                         1.466e+05
Df Model:                              33                                         
Covariance Type:                      HC3                                         
                                                                                                            coef    std 